# Enriquecimento — Lista de Espécies C1

Visualização do dataframe enriquecido com dados de **Flora do Brasil**, **IUCN** e **GBIF**.

Colunas preenchidas automaticamente quando estavam vazias:
- Taxonomia (`REINO`, `FILO`, `CLASSE`, `ORDEM`, `FAMÍLIA`, `GÊNERO`) — **GBIF**
- `GRAU DE AMEAÇA` — **IUCN Red List**
- `ENDEMISMO` (plantas) — **Flora e Funga do Brasil**

> Rode `poetry run python scripts/enrich_dataframe.py` antes para gerar/atualizar o CSV.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path().resolve().parent
ORIGINAL = ROOT / 'data' / 'Lista de Especies Hydro C1_teste.xlsx'
ENRICHED = ROOT / 'data' / 'C1_enriquecido.csv'

df_orig = pd.read_excel(ORIGINAL, sheet_name='C1')
df = pd.read_csv(ENRICHED)

print(f'Original:    {len(df_orig)} linhas')
print(f'Enriquecido: {len(df)} linhas')
df.head()

## Cobertura por coluna

Comparativo antes/depois de valores preenchidos.

In [ ]:
cols_target = ['REINO', 'FILO', 'CLASSE', 'ORDEM', 'FAMÍLIA', 'GÊNERO',
               'GRAU DE AMEAÇA', 'ENDEMISMO']

cov = pd.DataFrame({
    'antes_preenchidos': df_orig[cols_target].notna().sum(),
    'depois_preenchidos': df[cols_target].notna().sum(),
    'total_linhas': len(df),
})
cov['novos_preenchidos'] = cov['depois_preenchidos'] - cov['antes_preenchidos']
cov['cobertura_%'] = (cov['depois_preenchidos'] / cov['total_linhas'] * 100).round(1)
cov

## Distribuição de `GRAU DE AMEAÇA` (categorias IUCN)

In [ ]:
df['GRAU DE AMEAÇA'].value_counts(dropna=False)

## Distribuição de `ENDEMISMO`

In [ ]:
df['ENDEMISMO'].value_counts(dropna=False)

## Composição taxonômica

In [ ]:
display(df['REINO'].value_counts(dropna=False).to_frame('linhas'))
display(df['CLASSE'].value_counts(dropna=False).head(10).to_frame('linhas'))
display(df['FAMÍLIA'].value_counts(dropna=False).head(10).to_frame('linhas'))

## Espécies ameaçadas (categorias diferentes de LC/NE)

In [ ]:
ameacadas = df[~df['GRAU DE AMEAÇA'].isin(['LC', 'NE']) & df['GRAU DE AMEAÇA'].notna()]
ameacadas[['ESPÉCIE', 'FAMÍLIA', 'GRAU DE AMEAÇA', 'PONTO']].drop_duplicates().sort_values('GRAU DE AMEAÇA')

## Espécies endêmicas (plantas)

In [ ]:
endemicas = df[(df['ENDEMISMO'] == 'S') & (df['REINO'] == 'Plantae')]
endemicas[['ESPÉCIE', 'FAMÍLIA', 'GRAU DE AMEAÇA', 'PONTO']].drop_duplicates()

## Ocorrências por ponto

In [ ]:
df.groupby('PONTO').agg(
    linhas=('ESPÉCIE', 'size'),
    especies_unicas=('ESPÉCIE', 'nunique'),
).sort_values('especies_unicas', ascending=False).head(15)